# Run a population comparison

Edit the paths in the **settings** cell, then run all cells.

Leave `TEST_PATH` empty to use the training table for every test. `MODEL_PATH` is a pickled LightGBM estimator (`.pkl`). Relative paths resolve from the repository root (the directory that contains this notebook). Outputs go under `runs/`, which is gitignored.

After a successful run, open `runs/report.html` (plots embedded) or `runs/report.md` (plots in `runs/report_plots/`). Technical detail is in [docs/reference.md](docs/reference.md).


In [ ]:
import json
import logging
from pathlib import Path

from run_drift import DriftRun, DriftRunSettings, set_working_directory

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
ROOT = set_working_directory()
print(f"Working directory: {ROOT}")


## Settings

A relative path is resolved from the repository root. `MODEL_PATH` must be a `.pkl` (pickle or joblib) LightGBM model. Ten rounds are enough to see the pipeline; a decision uses 30.


In [ ]:
TRAIN_PATH = "demo_data/train.csv"
TEST_PATH = "demo_data/evaluation.csv"
NEW_PATH = "demo_data/new.csv"

FEATURES_FILE = "demo_data/features.txt"
CATEGORICAL = "c01,c02,c03,c04"
LABEL_COL = "target"
MODEL_PATH = "demo_data/model.pkl"
MISSING_SENTINELS = ""
OUTPUT_DIR = "runs/comparison"
REPORT_PATH = "runs/report.html"
N_ROUNDS = 10
THRESHOLD = 0.5
PERFORMANCE_TOLERANCE = 0.02
PROBLEM_TYPE = "binary"

settings = DriftRunSettings(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    new_path=NEW_PATH,
    features_file=FEATURES_FILE,
    categorical=CATEGORICAL,
    label_col=LABEL_COL,
    model_path=MODEL_PATH,
    missing_sentinels=MISSING_SENTINELS,
    output_dir=OUTPUT_DIR,
    report_path=REPORT_PATH,
    n_rounds=N_ROUNDS,
    threshold=THRESHOLD,
    performance_tolerance=PERFORMANCE_TOLERANCE,
    problem_type=PROBLEM_TYPE,
)


## Compare

This writes `decision.json` under `OUTPUT_DIR`, the HTML report to `REPORT_PATH`, and a markdown copy with PNG plots beside it.


In [ ]:
exit_code = DriftRun(settings).run()
if exit_code != 0:
    raise RuntimeError(f"Comparison exited with code {exit_code}.")


## Verdict


In [ ]:
decision_path = Path(settings.output_dir) / "decision.json"
if not decision_path.is_file():
    raise FileNotFoundError(f"Comparison finished without {decision_path}.")
decision = json.loads(decision_path.read_text(encoding="utf-8"))
print(f"Verdict: {decision.get('verdict_level')}")
print(decision.get("verdict"))
html_report = Path(settings.report_path).resolve()
print(f"HTML report: {html_report}")
print(f"Markdown report: {html_report.with_suffix('.md')}")
